## Imports

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score)
from sklearn.feature_selection import SelectKBest, f_classif, RFE

df = pd.read_csv('../data/df_model.csv', sep=';', dtype={'CODE_INSEE': str})
display(df.head(5))
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df['Résultat'].value_counts())
print(f"\nProportion :")
print(df['Résultat'].value_counts(normalize=True))

,Année,Code_INSEE,Résultat,pct_Femmes,pct_Agriculteurs,pct_Artisans,pct_Cadres,pct_Intermédiaires,pct_Employés,pct_Ouvriers,...,pct_25-39 ans,pct_40-54 ans,pct_55-64 ans,pct_80 ans et +,pct_Mariés,pct_Pacsés,pct_Concubinage,pct_Divorcés,pct_Célibataires,Population_active
0,2022,01001,droite,47.184259,0.704243,5.633941,10.563640,14.789096,15.493339,17.606067,...,21.664275,25.394548,18.794835,6.312769,51.219512,7.890961,16.499283,3.156385,15.351506,697.000000
1,2022,01002,gauche,41.363637,11.489899,2.297980,9.191919,13.787879,9.191919,9.191919,...,18.981481,34.259259,12.037037,8.333333,47.685185,11.111111,9.722222,7.407407,20.833333,216.000000
2,2022,01004,droite,52.881826,0.127297,3.114317,7.806622,15.437689,17.760558,15.886919,...,24.608657,21.861014,14.436934,6.106620,40.957640,5.835323,9.955739,8.566899,27.644693,12594.508695
3,2022,01005,droite,51.346262,0.362612,6.556032,8.691036,15.289281,21.150485,13.928360,...,25.029362,25.784335,17.295346,5.562916,48.809515,7.840915,14.193995,4.509098,20.022831,1554.355560
4,2022,01006,droite,39.860139,0.000000,0.000000,0.000000,9.965035,14.947553,44.842658,...,21.153846,17.307692,23.076923,5.769231,49.038462,8.653846,10.576923,9.615385,19.230769,104.000000


Dataset chargé : 30438 lignes, 28 colonnes

Distribution de la cible (vote politique) :
Résultat
droite    23386
centre     4240
gauche     2812
Name: count, dtype: int64

Proportion :
Résultat
droite    0.768316
centre    0.139300
gauche    0.092385
Name: proportion, dtype: float64


## Split test-train 30-70

In [20]:
X = df.drop(['Année', 'Code_INSEE', 'Résultat', 'Population_active', 'Population avec enfants'], axis=1)
y = df['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

✓ Données préparées
  Train: (21306, 23)
  Test: (9132, 23)


## Pipeline basique - comparaison de modèles

In [21]:
# Fonction pour entraînement et prédiction
def train_and_predict(pipeline):
    pipeline.fit(X_train, y_train)
    print(f"✓ Pipeline entraîné !")


    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Précision : {accuracy:.2%}")


# Fonction pour détecter l'overfitting
def detect_overfitting(pipeline):
    pipeline.fit(X_train, y_train)

    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    gap = train_score - test_score


    print(f"  Train : {train_score:.2%}")
    print(f"  Test  : {test_score:.2%}")
    print(f"  Gap   : {gap:.2%} ", end="")


    if gap > 0.10:
        print("⚠️ OVERFITTING détecté !")
    elif test_score < 0.75:
        print("⚠️ UNDERFITTING détecté !")
    else:
        print("✅ Bon équilibre")


### Random Forest

In [22]:
pipeline_forest = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])
detect_overfitting(pipeline_forest)


  Train : 98.24%
  Test  : 77.73%
  Gap   : 20.52% ⚠️ OVERFITTING détecté !


### Régression logistique

In [23]:
pipeline_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])


detect_overfitting(pipeline_logistic)

  Train : 77.88%
  Test  : 77.74%
  Gap   : 0.15% ✅ Bon équilibre


### Arbre de décision

In [24]:
pipeline_tree = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


detect_overfitting(pipeline_tree)

  Train : 100.00%
  Test  : 66.33%
  Gap   : 33.67% ⚠️ OVERFITTING détecté !


## Observations

__Random Forest et Decision Tree ont 100 % de précision sur le train.
Les paramètres n'étant pas ajustés, ils arrivent à identifier une commune d'après le nombre d'habitants, etc.__

Solutions :
- Ajustement des paramètres

OU

- Transformer le nombre d'habitants en variable catégorielle (village, ville moyenne, grande ville, ville principale, etc.) + transformer le nombre d'inscrits, etc. en pourcentage par rapport à la population totale


**Pour l'optimisation des modèles :** Possibilité de s'y mettre à 3, chacun essaye d'optimiser un modèle.

## Optimisation basique de Logistic Regression

### Validation croisée

In [25]:
# Fonction pour la validation croisée
def get_cross_val_scores(pipeline):
    cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

    print("Scores de validation croisée (5-fold) :")
    print(f"  Scores : {[f'{score:.2%}' for score in cv_scores]}")
    print(f"  Moyenne : {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})")

get_cross_val_scores(pipeline_logistic)

Scores de validation croisée (5-fold) :
  Scores : ['77.00%', '77.56%', '77.30%', '77.56%', '78.12%']
  Moyenne : 77.51% (+/- 0.37%)


### Sélection de features

In [26]:
pipeline_selection = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', LogisticRegression(random_state=42))
])

train_and_predict(pipeline_selection)


selector = pipeline_selection.named_steps['selector']
features_selected = X.columns[selector.get_support()].tolist()
print("\nFeatures sélectionnées :")
for i, feature in enumerate(features_selected):
    print(f"   {i+1}- {feature}")

✓ Pipeline entraîné !
Précision : 77.49%

Features sélectionnées :
   1- pct_Cadres
   2- pct_Ouvriers
   3- pct_Personne seule
   4- pct_Mariés
   5- pct_Célibataires


In [27]:
# Comparaison avec différents nombres de features
# k_values = [5, 10, 15, 20, 25, 30, 35, 40]
# results = []

# for k in k_values:
#     pipeline = Pipeline([
#         ('scaler', StandardScaler()),
#         ('selector', SelectKBest(f_classif, k=k)),
#         ('classifier', RandomForestClassifier(random_state=42))
#     ])
    
#     cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
#     results.append({
#         'k': k,
#         'mean_accuracy': cv_scores.mean(),
#         'std_accuracy': cv_scores.std()
#     })

# # Affichage des résultats
# results_df = pd.DataFrame(results)
# print("Résultats selon le nombre de features :")
# print(results_df.to_string(index=False))